# LangGraph V1.0 시작하기

이 노트북에서는 LangGraph V1.0의 핵심 기능들과 Agent 실습을 배워봅니다.

**학습 목표:**
- LangGraph 에이전트 생성 및 설정
- 도구(Tool) 정의 및 활용
- 컨텍스트 관리와 상태 유지
- 구조화된 응답 생성
- 메모리를 통한 대화 기록 관리

**실습 도메인:** 사내 포탈 업무 도우미 챗봇

## 1. 환경 설정

먼저 필요한 환경 변수를 설정합니다. `.env` 파일에 API 키를 추가해야 합니다.

```env
GOOGLE_API_KEY=your-key-here
LANGCHAIN_API_KEY=your-key-here
LANGCHAIN_TRACING_V2=true
```

In [ ]:
import sys
import asyncio

# Windows: ProactorEventLoop는 zmq/httpx의 add_reader를 지원하지 않아
# 커널이 멈춘 것처럼 보이는 문제가 있어 SelectorEventLoop로 강제 전환합니다.
if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())


In [ ]:
from dotenv import load_dotenv

# 환경 변수 로드
load_dotenv(override=True)

In [ ]:
from utils import logging

# LangSmith 추적 시작
logging.langsmith("01-langgraph-agent")

## 2. 모델 초기화

LangChain의 `init_chat_model` 함수를 사용하면 다양한 LLM 제공자의 모델을 통합된 방식으로 초기화할 수 있습니다.

### 모델 지정 방식

**기본 방식:** 모델 이름만 지정
```python
"gpt-4"
"claude-3-sonnet"
```

**통합 방식:** 제공자와 모델을 함께 지정 (이 노트북은 Gemini 무료 티어를 사용합니다)
```python
"google_genai:gemini-flash-lite-latest"
"openai:gpt-4"
"anthropic:claude-3-sonnet"
"azure_openai:gpt-4"
```

### 주요 매개변수
- **temperature**: 출력의 창의성 조절 (0.0 = 결정적, 1.0 = 창의적)
- **max_tokens**: 생성할 최대 토큰 수
- **timeout**: 응답 대기 시간 (초)
- **max_retries**: 실패 시 재시도 횟수

In [ ]:
from langchain.chat_models import init_chat_model

# Gemini Flash-Lite 모델 초기화 (무료 티어 할당량이 넉넉함, 온도 0으로 일관된 응답 생성)
llm = init_chat_model("google_genai:gemini-flash-lite-latest", temperature=0, max_tokens=2000)

In [ ]:
from utils.messages import stream_response

# 모델 테스트
result = llm.stream("AI Agent가 무엇인지 한 문장으로 설명해주세요.")
stream_response(result)

## 3. 기본 에이전트 생성

LangGraph V1.0에서는 `create_agent` 함수를 사용하여 에이전트를 생성합니다.

에이전트는 사용자의 요청을 이해하고, 필요한 도구를 선택하며, 적절한 응답을 생성하는 지능형 시스템입니다.

In [ ]:
from langchain.agents import create_agent

# 도구 없이 기본 에이전트 생성
basic_agent = create_agent(
    model="google_genai:gemini-flash-lite-latest",
    tools=[],
    system_prompt="""당신은 전문적인 사내 포탈 업무 도우미입니다. 
    임직원의 질문에 친절하고 정확하게 답변하세요.
    필요한 경우 제공된 도구를 사용하여 정보를 조회하세요.
    확인 할 수 없는 정보에는 "죄송합니다. 해당 정보를 확인 할 수 없습니다.""",
)

### 에이전트 그래프 시각화

에이전트의 내부 구조를 시각화하여 데이터 흐름을 이해할 수 있습니다.

In [ ]:
from utils.graphs import visualize_graph

visualize_graph(basic_agent)

In [ ]:
from utils.messages import stream_graph
from langchain_core.messages import HumanMessage

# 기본 대화 테스트
stream_graph(
    basic_agent,
    inputs={"messages": [HumanMessage(content="안녕하세요! 당신은 누구인가요?")]},
)

In [ ]:
from utils.messages import stream_graph
from langchain_core.messages import HumanMessage

# 기본 대화 테스트
stream_graph(
    basic_agent,
    inputs={"messages": [HumanMessage(content="사내 포탈의 근태/휴가 카테고리에 어떤 업무들이 있나요?")]},
)

In [ ]:
stream_graph(
    basic_agent,
    inputs={
        "messages": [
            HumanMessage(content="신청번호 REQ-2024-001의 처리 상태를 알려주세요.")
        ]
    },
)

## 4. 도구(Tool) 정의하기

도구는 에이전트가 외부 시스템이나 데이터와 상호작용할 수 있게 해주는 함수입니다.

### 실습: 사내 포탈 업무 도우미 도구

사내 포탈 업무 도우미를 위한 세 가지 도구를 만들어봅니다:
1. **업무 검색**: 카테고리별 사내 업무 찾기
2. **신청 조회**: 신청 번호로 처리 상태 확인
3. **추천 받기**: 임직원 상황 기반 관련 업무 추천

### 데이터 베이스 생성

In [ ]:
from typing import TypedDict, List

class InternalJob(TypedDict):
    """사내 업무 정보 스키마."""
    name: str
    department: str
    description: str

class Request(TypedDict):
    """신청 처리 상태 스키마."""
    status: str
    item: str
    processed_date: str

# 가상의 사내 업무 데이터베이스
INTERNAL_JOB_DB: dict[str, List[InternalJob]] = {
    "조직/인사": [
        {"name": "인사발령 조회", "department": "인사팀", "description": "본인의 인사발령 내역 및 발령일자를 조회합니다."},
        {"name": "조직도 조회", "department": "인사팀", "description": "회사 전체 조직 및 부서별 구성원을 조회합니다."},
    ],
    "근태/휴가": [
        {"name": "연차 신청", "department": "인사팀", "description": "연차휴가를 신청합니다."},
        {"name": "재택근무 신청", "department": "인사팀", "description": "재택근무 대상자는 재택근무를 신청할 수 있습니다."},
    ],
    "복리후생": [
        {"name": "복지포인트 사용", "department": "총무팀", "description": "연간 지급된 복지포인트의 잔액 및 사용내역을 조회합니다."},
        {"name": "건강검진 신청", "department": "총무팀", "description": "임직원 건강검진을 신청합니다."},
    ],
    "PC/IT환경": [
        {"name": "VPN 접속 신청", "department": "IT지원팀", "description": "외부에서 사내 시스템에 접속하기 위한 VPN 사용을 신청합니다."},
        {"name": "PC 초기 설정", "department": "IT지원팀", "description": "신규 PC 지급 후 회사 표준 환경을 설정합니다."},
    ],
    "보안/정보보호": [
        {"name": "비밀번호 변경", "department": "정보보안팀", "description": "사내 계정 비밀번호를 변경합니다."},
    ],
}

# 신청 처리 데이터베이스
REQUEST_DB: dict[str, Request] = {
    # 승인완료 신청
    "REQ-2024-001": {
        "status": "승인완료",
        "item": "연차 신청",
        "processed_date": "2024-12-20"
    },
    "REQ-2024-002": {
        "status": "승인완료",
        "item": "재택근무 신청",
        "processed_date": "2024-12-18"
    },
    "REQ-2024-003": {
        "status": "승인완료",
        "item": "VPN 접속 신청",
        "processed_date": "2024-12-15"
    },
    # 검토중 신청
    "REQ-2024-004": {
        "status": "검토중",
        "item": "경조금 신청",
        "processed_date": "2025-01-05"
    },
    "REQ-2024-005": {
        "status": "검토중",
        "item": "출장 신청",
        "processed_date": "2025-01-06"
    },
    # 대기중 신청
    "REQ-2025-001": {
        "status": "대기중",
        "item": "업무시스템 권한 신청",
        "processed_date": "2025-01-10"
    },
    "REQ-2025-002": {
        "status": "대기중",
        "item": "공유폴더 접근권한 신청",
        "processed_date": "2025-01-08"
    },
    # 반려 신청
    "REQ-2025-003": {
        "status": "반려",
        "item": "외부 저장매체 사용 신청",
        "processed_date": "2025-01-09"
    },
}


### Tools 생성

In [ ]:
from langchain.tools import tool
from typing import Literal

@tool
def get_all_job() -> str:
    """전체 사내 업무 카탈로그를 조회합니다.

    모든 카테고리의 전체 업무 목록을 반환합니다.
    LLM이 전체 업무를 파악하고 임직원 상황에 맞는 업무를 직접 선택할 수 있습니다.

    Returns:
        전체 카테고리별 업무 목록 (업무명, 담당부서, 설명 포함)
    """
    result = "=== 전체 사내 업무 카탈로그 ===\n\n"

    for category, jobs in INTERNAL_JOB_DB.items():
        result += f"【{category}】\n"
        for j in jobs:
            result += f"  - {j['name']} ({j['department']}): {j['description']}\n"
        result += "\n"

    result += f"총 {sum(len(jobs) for jobs in INTERNAL_JOB_DB.values())}개 업무"
    return result

@tool
def search_job(
    category: Literal["조직/인사", "근태/휴가", "복리후생", "PC/IT환경", "보안/정보보호"]
) -> str:
    """특정 카테고리의 사내 업무를 검색합니다.

    지정된 카테고리의 모든 업무 정보를 반환합니다.
    LLM이 카테고리 내 업무들을 분석하여 임직원에게 적합한 업무를 안내할 수 있습니다.

    Args:
        category: 검색할 업무 카테고리 (조직/인사, 근태/휴가, 복리후생, PC/IT환경, 보안/정보보호)

    Returns:
        해당 카테고리의 업무 목록 (업무명, 담당부서, 설명 포함)
    """
    jobs = INTERNAL_JOB_DB.get(category, [])

    if not jobs:
        return f"{category} 카테고리에 업무가 없습니다."

    result = f"【{category}】 카테고리 업무 목록:\n\n"
    for j in jobs:
        result += f"- {j['name']}\n"
        result += f"  담당부서: {j['department']}\n"
        result += f"  설명: {j['description']}\n\n"

    result += f"--- 카테고리 통계 ---\n"
    result += f"업무 수: {len(jobs)}개\n"

    return result


@tool
def search_portal_by_keyword(keyword: str) -> str:
    """키워드로 전체 카테고리에서 사내 업무를 검색합니다.

    입력된 키워드가 업무명에 포함된 모든 업무를 검색합니다.
    LLM이 검색 결과를 분석하여 임직원 상황에 맞는 업무를 선택할 수 있습니다.

    Args:
        keyword: 검색할 키워드 (예: "연차", "VPN", "비밀번호")

    Returns:
        키워드가 포함된 업무 목록 (카테고리, 업무명, 담당부서, 설명 포함)
    """
    results = []

    for category, jobs in INTERNAL_JOB_DB.items():
        for j in jobs:
            if keyword.lower() in j['name'].lower():
                results.append({
                    'category': category,
                    'name': j['name'],
                    'department': j['department'],
                    'description': j['description']
                })

    if not results:
        return f"'{keyword}' 키워드로 검색된 업무가 없습니다."

    result = f"'{keyword}' 검색 결과:\n\n"
    for item in results:
        result += f"- {item['name']} [{item['category']}]\n"
        result += f"  담당부서: {item['department']}\n"
        result += f"  설명: {item['description']}\n\n"

    result += f"총 {len(results)}개 업무 검색됨"
    return result


@tool
def check_request_status(request_id: str) -> str:
    """신청 번호로 처리 상태를 조회합니다.

    Args:
        request_id: 신청 번호 (예: REQ-2024-001)

    Returns:
        신청 건의 현재 처리 상태
    """
    request = REQUEST_DB.get(request_id)

    if not request:
        return f"신청번호 {request_id}를 찾을 수 없습니다."

    return (
        f"신청번호: {request_id}\n"
        f"업무: {request['item']}\n"
        f"상태: {request['status']}\n"
        f"처리예정일: {request['processed_date']}"
    )


### 도구를 갖춘 에이전트 생성

In [ ]:
PORTAL_TOOLS = [
    get_all_job,
    search_job,
    search_portal_by_keyword,
    check_request_status,
]

# 도구를 장착한 에이전트 생성
portal_agent = create_agent(
    model=llm,
    tools=PORTAL_TOOLS,
    system_prompt="""당신은 전문적인 사내 포탈 업무 도우미입니다. 
    임직원의 질문에 친절하고 정확하게 답변하세요.
    필요한 경우 제공된 도구를 사용하여 정보를 조회하세요.
    확인 할 수 없는 정보에는 "죄송합니다. 해당 정보를 확인 할 수 없습니다.""",
)

In [ ]:
# 그래프 시각화 - 도구가 추가된 것을 확인
visualize_graph(portal_agent)

In [ ]:
# 업무 검색 테스트
stream_graph(
    portal_agent,
    inputs={"messages": [HumanMessage(content="사내 포탈의 근태/휴가 카테고리에 어떤 업무들이 있나요?")]},
)

In [ ]:
# 신청 처리상태 조회 테스트
stream_graph(
    portal_agent,
    inputs={
        "messages": [
            HumanMessage(content="신청번호 REQ-2024-001의 처리 상태를 알려주세요.")
        ]
    },
)

In [ ]:
# 상황 기반 추천 테스트
stream_graph(
    portal_agent,
    inputs={
        "messages": [
            HumanMessage(content="다음 주에 다른 부서로 이동하는데 어떤 업무를 확인해야 할까요?")
        ]
    },
)

## 5. 컨텍스트(Context) 관리

컨텍스트를 사용하면 도구가 실행 시점의 추가 정보에 접근할 수 있습니다.

### 실습: 임직원 프로필 기반 개인화

임직원의 부서/직급과 근속연수를 컨텍스트로 전달하여 개인화된 복지 혜택 안내를 제공합니다.

In [ ]:
from dataclasses import dataclass
from langchain.tools import ToolRuntime
from typing import List

# 임직원 데이터베이스
EMPLOYEE_DB = {
    "yunah": {
        "name": "김윤아",
        "department": "인사팀",
        "position": "대리",
        "years_of_service": 4,
    },
    "wooyeol": {
        "name": "박우열",
        "department": "IT지원팀",
        "position": "사원",
        "years_of_service": 1,
    },
    "haseom": {
        "name": "신하섬",
        "department": "경영지원본부",
        "position": "과장",
        "years_of_service": 3,
    },
}


@dataclass
class EmployeeContext:
    """임직원 컨텍스트 정보"""

    employee_id: str


@tool
def get_employee_profile(runtime: ToolRuntime[EmployeeContext]) -> str:
    """현재 임직원의 프로필 정보를 조회합니다."""
    employee_id = runtime.context.employee_id

    employee = EMPLOYEE_DB.get(employee_id)

    if not employee:
        return "임직원 정보를 찾을 수 없습니다."

    return (
        f"이름: {employee['name']}\n"
        f"부서: {employee['department']}\n"
        f"직급: {employee['position']}\n"
        f"근속연수: {employee['years_of_service']}년"
    )


@tool
def get_welfare_point_policy() -> str:
    """근속연수별 복지포인트 지급 정책을 조회 합니다."""

    policy = {
        "3년 이상": "연 150만 포인트 지급 + 건강검진 고급형",
        "1년 이상 3년 미만": "연 100만 포인트 지급 + 건강검진 기본형",
        "1년 미만": "연 50만 포인트 지급",
    }

    return f"근속연수별 복지포인트:\n{policy}"

In [ ]:
# 컨텍스트를 사용하는 에이전트 생성
personalized_agent = create_agent(
    model=llm,
    tools=[get_employee_profile, get_welfare_point_policy, search_job],
    context_schema=EmployeeContext,
    system_prompt="""당신은 임직원 개인화 사내 포탈 서비스를 제공하는 전문 상담사입니다.
    임직원의 프로필과 복지포인트 정책을 조회하여 최적의 안내를 제공하세요.""",
)

In [ ]:
# 근속연수 3년 임직원으로 테스트
stream_graph(
    personalized_agent,
    inputs={"messages": [HumanMessage(content="제 프로필과 복지포인트 혜택을 알려주세요.")]},
    context=EmployeeContext(employee_id="haseom"),
)

In [ ]:
# 신입 임직원으로 테스트
stream_graph(
    personalized_agent,
    inputs={
        "messages": [HumanMessage(content="저는 어떤 복지포인트 혜택을 받을 수 있나요?")]
    },
    context=EmployeeContext(employee_id="wooyeol"),
)

## 6. 구조화된 응답 형식

Pydantic 모델을 사용하여 에이전트의 응답을 특정 스키마에 맞게 구조화할 수 있습니다.

### 실습: 업무 처리 가이드 리포트 생성

In [ ]:
from pydantic import BaseModel, Field
from typing import List


class JobGuide(BaseModel):
    """업무 처리 가이드 리포트 스키마"""

    job_name: str = Field(description="업무명")
    category: str = Field(description="업무 카테고리")
    department: str = Field(description="담당부서")
    required_documents: List[str] = Field(description="필요서류 목록")
    procedure: str = Field(description="처리절차")
    cautions: List[str] = Field(description="유의사항 목록")
    recommendation: str = Field(description="안내 의견")

In [ ]:
# 구조화된 응답을 사용하는 에이전트
structured_agent = create_agent(
    model=llm,
    tools=[search_job],
    response_format=JobGuide,
    system_prompt="""사내 업무 정보를 분석하여 구조화된 처리 가이드를 작성하세요.
    필요서류와 처리절차를 명확히 정리하고 구체적인 안내 의견을 제시하세요.""",
)

In [ ]:
# 구조화된 응답 테스트
response = structured_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="근태/휴가 카테고리에서 연차 신청에 대한 상세 가이드를 작성해주세요."
            )
        ]
    }
)

print("=== JSON 응답 ===")
print(response["messages"][-1].content)
print("\n=== 구조화된 객체 ===")
print(response["structured_response"])

## 7. 메모리와 대화 상태 관리

체크포인터(Checkpointer)를 사용하여 대화 히스토리를 저장하고 이전 대화를 기억할 수 있습니다.

### thread_id를 통한 대화 세션 관리

동일한 `thread_id`를 사용하면 같은 대화 컨텍스트를 유지할 수 있습니다.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# 메모리 체크포인터 생성
memory = InMemorySaver()

In [ ]:
# 메모리를 사용하는 에이전트
memory_agent = create_agent(
    model=llm,
    tools=[search_job, check_request_status],
    checkpointer=memory,
    system_prompt="당신은 임직원의 이전 대화를 기억하는 사내 포탈 업무 도우미입니다.",
)

# 첫 번째 대화 세션
config_session_1 = {"configurable": {"thread_id": "session_001"}}

In [ ]:
# 첫 번째 질문
stream_graph(
    memory_agent,
    inputs={
        "messages": [
            HumanMessage(
                content="안녕하세요, 저는 김윤아입니다. VPN 신청 방법을 찾고 있어요."
            )
        ]
    },
    config=config_session_1,
)

In [ ]:
# 두 번째 질문 (같은 세션)
stream_graph(
    memory_agent,
    inputs={
        "messages": [
            HumanMessage(
                content="제 이름이 뭔지 기억하시나요? 그리고 제가 찾던 업무는요?"
            )
        ]
    },
    config=config_session_1,
)

In [ ]:
# 다른 세션에서 질문 (새로운 thread_id)
config_session_2 = {"configurable": {"thread_id": "session_002"}}

stream_graph(
    memory_agent,
    inputs={"messages": [HumanMessage(content="제 이름이 뭔지 아시나요?")]},
    config=config_session_2,
)

## 정리

이 노트북에서 배운 LangGraph V1.0의 핵심 개념:

1. **모델 초기화**: `init_chat_model`로 다양한 LLM 통합
2. **에이전트 생성**: `create_agent`로 지능형 시스템 구축
3. **도구 정의**: `@tool` 데코레이터로 외부 기능 연동
4. **컨텍스트 관리**: `ToolRuntime`과 `context_schema`로 실행 환경 정보 전달
5. **구조화된 응답**: Pydantic 모델로 일관된 출력 형식 보장
6. **메모리 관리**: `InMemorySaver`와 `thread_id`로 대화 히스토리 유지